# Lab 05 — Source Preparation

This notebook performs **runtime/source preparation only**.

It intentionally contains **no `CREATE SCHEMA` or `CREATE VOLUME` DDL**.
Production schema and managed Volumes are created declaratively by the Bundle.

Runtime directories created here:
- `landing/station_status/`
- `reference/`
- `test_data/`


In [ ]:
dbutils.widgets.text("catalog", "dbr_dev", "01 Catalog")
dbutils.widgets.text("schema", "parvinbadalov", "02 Schema")
dbutils.widgets.text("volume_name", "lab05_lakeflow", "03 Reference volume")
dbutils.widgets.text(
    "streaming_volume_name",
    "lab05_lakeflow_streaming",
    "04 Streaming volume",
)
dbutils.widgets.text("seed_snapshot_count", "3", "05 Seed snapshot count")


In [ ]:
import json
import re
import time
from datetime import datetime, timezone

import requests

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
volume_name = dbutils.widgets.get("volume_name").strip()
streaming_volume_name = dbutils.widgets.get("streaming_volume_name").strip()
seed_snapshot_count = int(dbutils.widgets.get("seed_snapshot_count"))

_SAFE_IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")

for label, value in {
    "catalog": catalog,
    "schema": schema,
    "volume_name": volume_name,
    "streaming_volume_name": streaming_volume_name,
}.items():
    if not _SAFE_IDENTIFIER.fullmatch(value):
        raise ValueError(f"Unsafe {label}: {value!r}")

if seed_snapshot_count < 1:
    raise ValueError("seed_snapshot_count must be >= 1")

reference_dir = f"/Volumes/{catalog}/{schema}/{volume_name}/reference"
test_data_dir = f"/Volumes/{catalog}/{schema}/{volume_name}/test_data"
landing_dir = (
    f"/Volumes/{catalog}/{schema}/{streaming_volume_name}"
    "/landing/station_status"
)

STATION_INFORMATION_URL = (
    "https://gbfs.lyft.com/gbfs/2.3/bkn/en/station_information.json"
)
STATION_STATUS_URL = (
    "https://gbfs.lyft.com/gbfs/2.3/bkn/en/station_status.json"
)

print(f"Catalog          : {catalog}")
print(f"Schema           : {schema}")
print(f"Reference volume : {volume_name}")
print(f"Streaming volume : {streaming_volume_name}")
print(f"Seed snapshots   : {seed_snapshot_count}")


## Create runtime directories

These are normal filesystem directories inside already-existing Unity Catalog
Volumes. They are not database DDL objects.


In [ ]:
for path in (reference_dir, test_data_dir, landing_dir):
    dbutils.fs.mkdirs(path)
    print(f"✅ Directory ready: {path}")


## Download current station reference data

The batch/reference feed is written to the reference directory.


In [ ]:
def fetch_json(url: str) -> dict:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.json()


station_information = fetch_json(STATION_INFORMATION_URL)
station_information_path = f"{reference_dir}/station_information.json"

dbutils.fs.put(
    station_information_path,
    json.dumps(station_information),
    overwrite=True,
)

station_count = len(station_information.get("data", {}).get("stations", []))

print("✅ STATION INFORMATION READY")
print(f"Stations : {station_count:,}")
print(f"File     : {station_information_path}")


## Seed immutable station-status snapshots

Each API poll is written to a new timestamped file so Auto Loader can process
new files incrementally.


In [ ]:
created_files = []

for index in range(seed_snapshot_count):
    payload = fetch_json(STATION_STATUS_URL)
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    output_path = f"{landing_dir}/station_status_{timestamp}.json"

    dbutils.fs.put(
        output_path,
        json.dumps(payload),
        overwrite=False,
    )
    created_files.append(output_path)

    stations = len(payload.get("data", {}).get("stations", []))
    print(
        f"✅ Seed snapshot {index + 1}/{seed_snapshot_count}: "
        f"{stations:,} stations -> {output_path}"
    )

    if index + 1 < seed_snapshot_count:
        time.sleep(1)


## Validation


In [ ]:
required_paths = {
    "reference": reference_dir,
    "test_data": test_data_dir,
    "station_status_landing": landing_dir,
}

for label, path in required_paths.items():
    try:
        dbutils.fs.ls(path)
    except Exception as exc:
        raise RuntimeError(f"Required directory is unavailable: {label} -> {path}") from exc

if not created_files:
    raise RuntimeError("No station_status seed snapshots were created.")

print()
print("✅ LAB 05 SOURCE PREPARATION PASSED")
print(f"Reference file : {station_information_path}")
print(f"Seed files     : {len(created_files)}")
print(f"Landing path   : {landing_dir}")
